# E3 · CLIP congelado + cabeza MLP entrenada con Triplet Loss

**Pregunta.** ¿Aprender una proyección métrica especializada (MLP) sobre las representaciones visuales de E2 (YOLO crop + CLIP) mejora la separación entre mascotas parecidas respecto de CLIP zero-shot?

| | |
|---|---|
| Pipeline | YOLOv8n (crop 10%) → CLIP ViT-B/32 congelado → MLP 512→256→512 → L2 |
| Entrenamiento | Lotes $P \times K$ ($P=8, K=4$) por especie, Batch-Hard Triplet Loss (margen 0.2) |
| Datos de entrenamiento | Datasets públicos de cuerpo entero: MPDD (perros) y Cat Individuals (gatos) |
| Evaluación | `pettrace-reid` (fotos reales de mascotas en Lima, evaluación zero-shot independiente) |


## Entorno del experimento

Las dependencias y sus versiones se declaran aquí. `%pip` instala en el kernel seleccionado; ejecuta esta celda antes de los imports. Si actualiza paquetes ya cargados, reinicia el kernel y vuelve a ejecutar.

E3 conserva su entorno de entrenamiento con PyTorch (GPU para entrenamiento) y ONNX Runtime (CPU para evaluación); se importa PyTorch antes de ONNX Runtime.

In [ ]:
# Dependencias de evaluación y entrenamiento de E3.
%pip install -q --only-binary=:all: \
    "numpy>=2.1,<3" \
    "pillow>=11,<13" \
    "pyyaml>=6,<7" \
    "onnxruntime==1.20.1" \
    "mlflow-skinny==3.16.1" \
    "huggingface-hub>=1.32,<3" \
    "matplotlib>=3.9,<4" \
    "torch>=2.5,<3" \
    "onnx>=1.17,<2" \
    "pandas>=2.2.3,<4"

In [ ]:
# Install the package independently of the notebook's location.
import importlib.util
import os

PACKAGE_SOURCE = os.environ.get(
    "PETTRACE_PACKAGE_SOURCE", "git+https://github.com/alessandro54/pettrace-ml.git@main"
)
if importlib.util.find_spec("pettrace") is None:
    get_ipython().run_line_magic("pip", f'install -q "{PACKAGE_SOURCE}"')

import pettrace
print("pettrace-ml:", pettrace.__version__)

In [ ]:
# Credentials stay in environment variables / Secrets / hidden prompts, never in notebook source.
import getpass
import os

try:
    from google.colab import userdata
except ImportError:
    userdata = None

for name in ("HF_TOKEN", "MLFLOW_TRACKING_URI", "MLFLOW_TRACKING_USERNAME", "MLFLOW_TRACKING_PASSWORD"):
    if not os.environ.get(name) and userdata is not None:
        try:
            os.environ[name] = userdata.get(name)
        except Exception:
            pass  # VS Code may not expose Secrets; use a hidden prompt below.
    if not os.environ.get(name):
        value = ("https://ml.pettrace.app" if name == "MLFLOW_TRACKING_URI"
                 else getpass.getpass(f"{name}: "))
        if not value.strip():
            raise ValueError(f"Falta {name}.")
        os.environ[name] = value

In [ ]:
import json, os, sys

os.environ.setdefault("HF_HUB_DISABLE_PROGRESS_BARS", "1")
os.environ.setdefault("MLFLOW_ENABLE_ARTIFACTS_PROGRESS_BAR", "false")
os.environ.setdefault("TQDM_DISABLE", "1")

import warnings
warnings.filterwarnings("ignore")

from collections import Counter
from pathlib import Path

import mlflow
import numpy as np
import torch
import onnxruntime as ort
import yaml
from IPython.display import Image, Markdown, display
from PIL import Image as PILImage, ImageOps

from pettrace import data, metrics
from pettrace.e3 import Config, Head, fit, export_encoder, project
from pettrace.evaluation import evaluate, load_model
from pettrace.tracking import production_context, publish_evaluation, save_provenance, source_identity
from pettrace.figures import explain, metrics_table, save
from pettrace.public_data import audit_and_split, discover, feature_cache

print("código  :", os.environ.get("GIT_SHA", "?")[:7], "(sin cambios locales)" if os.environ.get("GIT_DIRTY") == "false" else "(¡cambios sin commit!)")
print("MLflow  :", os.environ.get("MLFLOW_TRACKING_URI", "https://ml.pettrace.app"))
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device  : {device} ({torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'})")

EXPERIMENT = "E3"
DATASET = "v3"
BASE_MODEL = "models:/pettrace-embedder/2"
REPORT = Path(os.environ.get("PETTRACE_OUTPUT_DIR", "reports")) / f"e3_{DATASET}"
LOG_TO_MLFLOW = os.environ.get("E3_LOG_TO_MLFLOW", "false").lower() == "true"
mlflow.set_tracking_uri(os.environ["MLFLOW_TRACKING_URI"])
_, tracking_experiment = production_context()
SOURCE_SHA256 = source_identity()["source_sha256"]

## 1. Datos

El protocolo separa estrictamente las identidades de entrenamiento de las de evaluación:
* **Entrenamiento y validación (públicos)**: Perros de **MPDD** (192 perros, CC BY 4.0) y gatos de **Cat Individual Images** (Kaggle). Se auditan duplicados y se separan particiones por identidad.
* **Evaluación de la tesis (privada)**: `pettrace-reid` en Hugging Face, nunca visto durante el entrenamiento.


In [ ]:
# 1.1 Dataset de evaluación privada de la tesis
ds = data.load(DATASET)
bucket = lambda x: "oscura" if x < 0.25 else "tenue" if x < 0.4 else "normal"
rows_eval = ["| Especie | Animales | Fotos | Fotos por animal | Luz normal / tenue / oscura | Particiones |", "|---|---|---|---|---|---|"]
for sp in ("cat", "dog"):
    ph = [p for p in ds.photos if p.species == sp]
    an = set(p.animal for p in ph)
    per = [len([p for p in ph if p.animal == a]) for a in an]
    splits = Counter(p.split for p in ph)
    lights = Counter(bucket(p.light) for p in ph)
    rows_eval.append(
        f"| {sp} | {len(an)} | {len(ph)} | {min(per)}–{max(per)} (media {np.mean(per):.1f}) | "
        f"{lights['normal']} / {lights['tenue']} / {lights['oscura']} | "
        f"{splits['test']} test / {splits.get('val', 0)} val / {splits.get('train', 0)} train |"
    )
display(Markdown("\n".join(rows_eval)))


In [ ]:
# 1.2 Preparación y auditoría de datos públicos de entrenamiento
DATA_ROOT = Path("datasets/public")
DATA_ROOT.mkdir(parents=True, exist_ok=True)

from pettrace.acquisition import prepare
manifest_path, manifest_rows, audit_summary = prepare(DATA_ROOT)

print(f"Fotos públicas retenidas tras auditoría: {audit_summary['retained_photos']} de {audit_summary['raw_photos']}")
display(Markdown(f"""
| Especie | Split | Identidades | Fotos |
|---|---|---|---|
| Gatos | train | {audit_summary['counts']['cat.train']['identities']} | {audit_summary['counts']['cat.train']['photos']} |
| Gatos | val | {audit_summary['counts']['cat.val']['identities']} | {audit_summary['counts']['cat.val']['photos']} |
| Perros | train | {audit_summary['counts']['dog.train']['identities']} | {audit_summary['counts']['dog.train']['photos']} |
| Perros | val | {audit_summary['counts']['dog.val']['identities']} | {audit_summary['counts']['dog.val']['photos']} |
"""))

## 2. Modelo Base (E2)

Cargamos el modelo base registrado de E2 (`models:/pettrace-embedder/2`), compuesto por el detector YOLOv8n (con 10% de margen) y el encoder CLIP ViT-B/32, y verificamos su *golden fingerprint*.


In [ ]:
base_model = load_model(BASE_MODEL)
assert base_model.golden_ok, "El modelo base E2 falló su prueba de huella dorada."
display(Markdown("\n".join([
    "| | |", "|---|---|",
    f"| URI Base | `{base_model.uri}` |",
    f"| Versión en Registro | {base_model.registry_version} |",
    f"| Pipeline | `{base_model.model_version}` |",
    *[f"| sha256 `{f}` | `{h}` |" for f, h in base_model.sha256.items()],
    f"| Golden Fingerprint | `{base_model.golden_fingerprint}` (OK) |",
])))

## 3. Entrenamiento (MLP Head + Batch-Hard Triplet Loss)

* La cabeza MLP proyecta: $512 \to 256 \to 512$ con ReLU y normalización $L_2$.
* Los lotes usan un muestreador $P \times K$ ($P=8$ identidades por lote, $K=4$ fotos por identidad), alternando lotes de gatos y de perros.
* Se cachean los embeddings extraídos por el E2 congelado para acelerar el entrenamiento en GPU.
* El mejor checkpoint se selecciona únicamente usando el `mAP` del conjunto de validación público.


In [ ]:
config = Config(
    seed=42,
    epochs=30,
    steps_per_epoch=100,
    identities_per_batch=8,
    images_per_identity=4,
    hidden=256,
    margin=0.2,
    learning_rate=1e-3,
    weight_decay=1e-4,
    patience=7
)

head = Head(config.hidden)
print("Arquitectura de la Cabeza:")
print(head)


In [ ]:
# Extracción y caché de características usando la base E2 congelada
cache_dir = DATA_ROOT / "cache"
training_rows = [row for row in manifest_rows if row["split"] in {"train", "val"}]
features, cache_digest = feature_cache(base_model.pipeline, training_rows, cache_dir, base_model.dir)
print(f"Características congeladas listas: {features.shape}, hash de caché: {cache_digest[:16]}")

# Entrenamiento de la cabeza MLP
os.makedirs(REPORT, exist_ok=True)
trained_head, train_summary = fit(features, training_rows, Path(REPORT), config=config, device=device)
print(f"Entrenamiento completado en epoch {train_summary['best_epoch']} con val macro mAP: {train_summary['best_val_mAP']:.4f}")

In [ ]:
# Fusión de la cabeza entrenada dentro del grafo ONNX de E2
model_dir = os.path.join(REPORT, "model")
onnx_hashes = export_encoder(base_model.dir, trained_head, model_dir, model_version="clip-vit-b32-crop-mlp@v1")
print("Modelo E3 fusionado y exportado exitosamente:")
for name, sha in onnx_hashes.items():
    print(f"  {name}: {sha}")


## 4. Evaluación, paso a paso

Cargamos el modelo fusionado E3 resultante y lo evaluamos sobre el benchmark privado de Lima (`pettrace-reid`), exactamente con el mismo protocolo paso a paso que E1 y E2.


In [ ]:
e3_model = load_model(model_dir)
assert e3_model.golden_ok, "El modelo E3 fusionado no superó la verificación golden."
ev = evaluate(ds, e3_model.pipeline, zero_shot=True)
E = ev.embeddings
print(f"Embeddings calculados: {E.shape}, norma media = {np.linalg.norm(E, axis=1).mean():.4f}")

In [ ]:
animals = np.array([p.animal for p in ds.photos])
species = np.array([p.species for p in ds.photos])
S = E @ E.T  # Similitud coseno entre todas las fotos

# Evaluación leave-one-out dentro de la misma especie
rows_eval_calc = []
for q in range(len(E)):
    gallery = np.where((species == species[q]) & (np.arange(len(E)) != q))[0]
    if len(set(animals[gallery])) < 2 or not (animals[gallery] == animals[q]).any():
        continue
    sub = S[q, gallery]
    sub_animals = animals[gallery]
    sub_species = species[gallery]

    unique_ids = []
    seen = set()
    for a in sub_animals:
        if a not in seen:
            seen.add(a)
            unique_ids.append(a)
    scores = np.array([sub[sub_animals == a].max() for a in unique_ids])
    order = np.argsort(-scores, kind="stable")
    ordered_ids = np.array(unique_ids)[order]
    rank = int(np.where(ordered_ids == animals[q])[0][0]) + 1

    # Average Precision
    photo_order = np.argsort(-sub, kind="stable")
    labels = (sub_animals[photo_order] == animals[q]).astype(int)
    k = np.arange(1, len(labels) + 1)
    prec = np.cumsum(labels) / k
    ap = float((prec * labels).sum() / max(1, labels.sum()))

    rows_eval_calc.append({"query": q, "animal": animals[q], "species": species[q], "rank": rank, "ap": ap})

ranks = np.array([r["rank"] for r in rows_eval_calc])
aps = np.array([r["ap"] for r in rows_eval_calc])

print("Resultados calculados a mano:")
print(f"  Rank-1: {np.mean(ranks <= 1)*100:.1f} %")
print(f"  Rank-5: {np.mean(ranks <= 5)*100:.1f} %")
print(f"  mAP   : {aps.mean():.3f}")

In [ ]:
expected_ranks = {row["query"]: row["rank"] for row in ev.result["test"]["per_query"]}
assert all(row["rank"] == expected_ranks[row["query"]] for row in rows_eval_calc)
assert np.isclose(aps.mean(), ev.result["test"]["global"]["mAP"])
print("Verificación del cálculo visible con pettrace.evaluation: OK")

## 5. Resultados y comparación con E1 y E2


In [ ]:
s3 = save(ev, e3_model, str(REPORT))
baseline_summaries = {}
for label, uri in {"E1": "models:/pettrace-embedder/1", "E2": BASE_MODEL}.items():
    previous_model = base_model if label == "E2" else load_model(uri)
    assert previous_model.golden_ok
    previous_ev = evaluate(ds, previous_model.pipeline, zero_shot=True)
    baseline_summaries[label] = save(previous_ev, previous_model, str(REPORT / f"baseline_{label.lower()}"))
display(Markdown(metrics_table({**baseline_summaries, "E3": s3})))

In [ ]:
summ = json.load(open(os.path.join(REPORT, "summary.json")))
for name in ("confusion", "rank1_por_animal", "distribucion_scores", "cmc", "matriz_similitud"):
    img_path = os.path.join(REPORT, f"{name}.png")
    if os.path.exists(img_path):
        display(Image(img_path, width=760))
        display(Markdown(explain(name, summ)))


## 6. Registro en MLflow


In [ ]:
REGISTER_E3 = False
if REGISTER_E3 and not LOG_TO_MLFLOW:
    raise ValueError("Activa LOG_TO_MLFLOW para registrar los artefactos de esta ejecución.")
save_provenance(REPORT, e3_model)
receipt = REPORT / "publication.json"
receipt.write_text(json.dumps({"published": False, "run_id": None}))
if LOG_TO_MLFLOW:
    run_id = publish_evaluation(ev, e3_model, REPORT, SOURCE_SHA256, experiment=EXPERIMENT,
                                expected_dataset_version=DATASET, expected_protocol="zero-shot")
    receipt.write_text(json.dumps({"published": True, "run_id": run_id}))
    print("MLflow run:", run_id)
    if REGISTER_E3:
        version = mlflow.register_model(f"runs:/{run_id}/report/model", "pettrace-embedder")
        client = mlflow.MlflowClient()
        client.set_model_version_tag("pettrace-embedder", version.version, "experiment", "E3")
        client.set_model_version_tag("pettrace-embedder", version.version, "status", "candidate")
        print("Versión E3 registrada:", version.version)
else:
    print("Resultados guardados en", REPORT, "· publicación desactivada.")

## 7. Conclusiones

* Registra aquí tus observaciones una vez completada la ejecución:
  * ¿Mejoró el mAP y el Rank-1 en gatos respecto a la caída observada en E2?
  * ¿Cómo se compara la separación entre positivos y negativos en la matriz de confusión respecto a E1 y E2?
  * ¿El modelo CANDIDATO supera el umbral de aceptación para despliegue?
